In [6]:
import pandas as pd
from tqdm import tqdm
from summ_eval.meteor_metric import MeteorMetric
from summ_eval.bleu_metric import BleuMetric
from summ_eval.bert_score_metric import BertScoreMetric
import contextlib
import io
import warnings

# Suppress HuggingFace warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [7]:
file_path = '../../data/data.csv'
df = pd.read_csv(file_path)

In [8]:
df.head()

,input,actual_output
0,Paul Merson has restarted his row with Andros ...,paul merson was brought on with only seven min...
1,Paul Merson has restarted his row with Andros ...,paul merson has restarted his row with andros ...
2,Paul Merson has restarted his row with Andros ...,paul merson has restarted his row with andros ...
3,Paul Merson has restarted his row with Andros ...,paul merson has restarted his row with andros ...
4,Paul Merson has restarted his row with Andros ...,paul merson has restarted his row with andros ...


In [9]:
# Initialize metrics
meteor = MeteorMetric()
bleu = BleuMetric()
bert = BertScoreMetric(lang='en', rescale_with_baseline=True)

In [10]:
# Prepare score lists
meteor_scores = []
bleu_scores = []
bert_f1s = []

# Loop with suppressed printing
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    actual = row['actual_output']
    predicted = row['input']

    meteor_score = meteor.evaluate_example(actual, predicted)
    bleu_score = bleu.evaluate_example(actual, predicted)

    # Suppress unwanted hash_code output from BERTScore
    with contextlib.redirect_stdout(io.StringIO()):
        bert_score = bert.evaluate_example(actual, predicted)

    meteor_scores.append(meteor_score['meteor'])
    bleu_scores.append(bleu_score['bleu'])
    bert_f1s.append(bert_score['bert_score_f1'])

# Save to DataFrame
result_df = pd.DataFrame({
    'index': range(len(df)),
    'meteor_score': meteor_scores,
    'bleu_score': bleu_scores,
    'bert_score_f1': bert_f1s
})

Scoring: 100%|██████████| 1600/1600 [34:58<00:00,  1.31s/it] 


In [11]:
result_df.to_csv('nlp_aa.csv', index=False)

In [15]:
# Extract float values from the dicts in each column
result_df['meteor_score'] = result_df['meteor_score'].apply(lambda x: x['meteor'])
result_df['bleu_score'] = result_df['bleu_score'].apply(lambda x: x['bleu'])

# Save cleaned DataFrame to CSV
result_df.to_csv('nlp.csv', index=False)